# Member 2: Trigram/Four-gram Models and Sentence Generation

**Assignment tasks covered:** 1.3 (top 10 n-grams for n = 3, 4), 1.7 (sentence probability), 1.8 (random sentence generation).

**Preprocessing:** identical to Member 1 (same sentence-ending markers and punctuation regex). Verified: both give 5,057,775 sentences and 77,760,019 tokens on the full corpus.

**Probability method:** maximum likelihood, same as Member 1's bigram table:
P(w_n | context) = count(context w_n) / outgoing_count(context), where outgoing_count = how often the context is followed by any word.

**Memory approach:** the full corpus has ~42M unique trigrams and ~45M unique four-grams, too many for Colab RAM. Words are split into 8 fixed groups (CRC32 hash); each pass counts only n-grams whose first word is in one group. N-grams with the same context always land in the same group, so counts and probabilities are exact.

Run the cells in order in Google Colab. The corpus must be in My Drive as `GPAC.txt`.

## 1. Setup: paths, preprocessing rules, streaming reader

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

import re, time, pickle, csv, math, random, sys, zlib, gc
from collections import Counter
from pathlib import Path

CORPUS_PATH = Path("/content/gdrive/MyDrive/GPAC.txt")
OUT = Path("/content/gdrive/MyDrive/GPAC_member2_results")
OUT.mkdir(parents=True, exist_ok=True)

# Copied exactly from Member 1's preprocessing
SENT_END = re.compile(r"፡፡|።|፧|\?|!+")
PUNCT_RE = re.compile(r"""[፡።፣፤፥፦፧፨?!,.;:()\[\]{}<>"'/%—–-]+""")

def stream_sentences(path, chunk_chars=1_000_000):
    """Read the file in chunks and yield one sentence (list of tokens) at a time."""
    buffer = ""
    with open(path, encoding="utf-8") as f:
        while True:
            chunk = f.read(chunk_chars)
            if not chunk:
                break
            buffer += chunk
            last_end = 0
            for m in SENT_END.finditer(buffer):
                tokens = PUNCT_RE.sub(" ", buffer[last_end:m.end()]).split()
                if tokens:
                    yield tokens
                last_end = m.end()
            buffer = buffer[last_end:]      # keep the unfinished sentence for the next chunk
    tokens = PUNCT_RE.sub(" ", buffer).split()
    if tokens:
        yield tokens

## 2. Full-corpus check and unigram counts (must match Member 1)

In [ ]:
start = time.time()
unigrams = Counter()
n_sentences = 0
for tokens in stream_sentences(CORPUS_PATH):
    n_sentences += 1
    unigrams.update(tokens)

n_tokens = sum(unigrams.values())
print(f"Sentences      : {n_sentences:,}   (Member 1: 5,057,775)")
print(f"Tokens         : {n_tokens:,}   (Member 1: 77,760,019)")
print(f"Unique unigrams: {len(unigrams):,}")
with open(OUT / "full_unigrams.pkl", "wb") as f:
    pickle.dump({"unigrams": unigrams, "sentences": n_sentences}, f)
print(f"Time: {(time.time()-start)/60:.1f} min")

## 3. Sample n-grams (n = 1 to 4) from one sentence

In [ ]:
example = PUNCT_RE.sub(" ", "ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።").split()
for n in range(1, 5):
    grams = [" ".join(example[i:i+n]) for i in range(len(example) - n + 1)]
    print(f"{n}-grams: {grams}")

## 4. Count trigrams and four-grams on the full corpus (8 groups)
Resumable: finished groups are saved to Drive and skipped on re-run.

In [ ]:
N_PARTS = 8
PART_DIR = OUT / "parts"
PART_DIR.mkdir(exist_ok=True)

# Fixed group number 0-7 for every word (stable across restarts)
group = {w: zlib.crc32(w.encode("utf-8")) % N_PARTS for w in unigrams}

def top_with_prob(counter, k=50):
    """Top k n-grams plus how often their context is followed by any word."""
    top = counter.most_common(k)
    needed = {g.rsplit(" ", 1)[0] for g, _ in top}
    outgoing = Counter()
    for g, c in counter.items():
        ctx = g.rsplit(" ", 1)[0]
        if ctx in needed:
            outgoing[ctx] += c
    return [(g, c, outgoing[g.rsplit(" ", 1)[0]]) for g, c in top]

start = time.time()
for part in range(N_PARTS):
    part_file = PART_DIR / f"part_{part}.pkl"
    if part_file.exists():
        print(f"Group {part}: already done, skipping")
        continue
    tri, four = Counter(), Counter()
    for tokens in stream_sentences(CORPUS_PATH):
        g = [group[t] for t in tokens]
        L = len(tokens)
        for i in range(L - 2):
            if g[i] == part:
                tri[" ".join(tokens[i:i+3])] += 1
                if i + 3 < L:
                    four[" ".join(tokens[i:i+4])] += 1
    result = {"tri_unique": len(tri), "tri_total": sum(tri.values()),
              "four_unique": len(four), "four_total": sum(four.values()),
              "tri_top": top_with_prob(tri), "four_top": top_with_prob(four)}
    with open(part_file, "wb") as f:
        pickle.dump(result, f)
    print(f"Group {part}: {result['tri_unique']:,} trigrams, {result['four_unique']:,} four-grams | "
          f"{(time.time()-start)/60:.1f} min")
    del tri, four
    gc.collect()
print("All groups done.")

## 5. Top 10 trigrams and four-grams (task 1.3)

In [ ]:
parts = [pickle.load(open(PART_DIR / f"part_{p}.pkl", "rb")) for p in range(N_PARTS)]

for order, name, n in [("tri", "trigrams", 3), ("four", "fourgrams", 4)]:
    unique = sum(p[order + "_unique"] for p in parts)
    total = sum(p[order + "_total"] for p in parts)
    top10 = sorted((x for p in parts for x in p[order + "_top"]), key=lambda x: -x[1])[:10]

    print(f"\nTop 10 {name}  (unique: {unique:,}, total: {total:,})")
    for g, c, out in top10:
        print(f"  {c:>8,}  {g}   P = {c:,} / {out:,} = {c/out:.4f}")

    with open(OUT / f"top10_{name}_full.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow([f"w{i}" for i in range(1, n + 1)] + ["count", "outgoing_count", "probability"])
        for g, c, out in top10:
            w.writerow(g.split(" ") + [c, out, round(c / out, 6)])

## 6. Sentence probability (task 1.7)
Models: MLE bigram/trigram/four-gram, Laplace four-gram, and linear interpolation (weights 0.4/0.3/0.2/0.1, re-normalised over the orders available at each position).

In [ ]:
test_sentences = [
    "ኢትዮጵያ በአፍሪካ ቀንድ ትገኛለች።",      # the given sentence
    "ትገኛለች ቀንድ በአፍሪካ ኢትዮጵያ።",      # same words, scrambled order
    "አዲስ አበባ የኢትዮጵያ ዋና ከተማ ናት።",   # a common, natural sentence
    "ነገር ግን ይህ ብቻ አይደለም።",           # frequent words
]

# Collect only the n-grams the test sentences need, in one pass
needed_grams, needed_ctx = set(), set()
for s in test_sentences:
    t = PUNCT_RE.sub(" ", s).split()
    for i in range(len(t)):
        for n in (2, 3, 4):
            if i - n + 1 >= 0:
                needed_grams.add(" ".join(t[i-n+1:i+1]))
                needed_ctx.add(" ".join(t[i-n+1:i]))
test_words = {w for g in needed_ctx for w in g.split()}

gram_count, ctx_out = Counter(), Counter()
for tokens in stream_sentences(CORPUS_PATH):
    L = len(tokens)
    for i in range(L - 1):
        if tokens[i] not in test_words:
            continue
        for n in (2, 3, 4):
            if i + n <= L:
                ctx = " ".join(tokens[i:i+n-1])
                if ctx in needed_ctx:
                    ctx_out[ctx] += 1
                    gram = " ".join(tokens[i:i+n])
                    if gram in needed_grams:
                        gram_count[gram] += 1

V = len(unigrams)
LAMBDAS = {4: 0.4, 3: 0.3, 2: 0.2, 1: 0.1}

def mle(context, word):
    if not context:
        return unigrams[word] / n_tokens
    ctx = " ".join(context)
    out = ctx_out[ctx]
    return gram_count[ctx + " " + word] / out if out else 0.0

def laplace(context, word):
    if not context:
        return (unigrams[word] + 1) / (n_tokens + V)
    ctx = " ".join(context)
    return (gram_count[ctx + " " + word] + 1) / (ctx_out[ctx] + V)

def interp(context, word):
    usable = [k for k in (1, 2, 3, 4) if len(context) >= k - 1]
    total_w = sum(LAMBDAS[k] for k in usable)
    p = 0.0
    for k in usable:
        part = laplace((), word) if k == 1 else mle(context[len(context)-(k-1):], word)
        p += LAMBDAS[k] / total_w * part
    return p

def log10_prob(sentence, prob_fn, n):
    t = PUNCT_RE.sub(" ", sentence).split()
    lp = 0.0
    for i in range(len(t)):
        p = prob_fn(tuple(t[max(0, i-(n-1)):i]), t[i])
        if p == 0:
            return float("-inf")
        lp += math.log10(p)
    return lp

s = test_sentences[0]
t = PUNCT_RE.sub(" ", s).split()
print(f"=== Word by word: {s} ===")
for i in range(len(t)):
    ctx = tuple(t[max(0, i-3):i])
    print(f"  P({t[i]} | {' '.join(ctx) or '-'}):  MLE-4gram = {mle(ctx, t[i]):.6f}   interpolated = {interp(ctx, t[i]):.6f}")

print("\n=== log10 P(sentence) ===")
print(f"  {'MLE-2':>8} {'MLE-3':>8} {'MLE-4':>8} {'Lap-4':>8} {'Interp':>8}   sentence")
with open(OUT / "sentence_probabilities_full.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["sentence", "MLE_bigram", "MLE_trigram", "MLE_fourgram", "Laplace_fourgram", "Interpolated"])
    for s in test_sentences:
        row = [log10_prob(s, mle, n) for n in (2, 3, 4)] + [log10_prob(s, laplace, 4), log10_prob(s, interp, 4)]
        print("  " + " ".join(f"{x:8.3f}" for x in row) + f"   {s}")
        w.writerow([s] + [round(x, 4) if x != float("-inf") else "P=0 (unseen n-gram)" for x in row])

### Why the given sentence gets probability 0 under MLE: words that follow ቀንድ

In [ ]:
after_qend = Counter()
for tokens in stream_sentences(CORPUS_PATH):
    for i in range(len(tokens) - 1):
        if tokens[i] == "ቀንድ":
            after_qend[tokens[i + 1]] += 1

print(f"'ቀንድ' appears {unigrams['ቀንድ']:,} times, followed by {len(after_qend):,} different words")
print(f"'ቀንድ ትገኛለች' count: {after_qend['ትገኛለች']}")
for w, c in after_qend.most_common(15):
    print(f"  {c:>6,}  ቀንድ {w}")

## 7. Random sentence generation (task 1.8)
Trained on every 10th sentence of the full corpus (~506k sentences) so the model fits in memory while covering the whole corpus.

In [ ]:
EVERY = 10
next_word = {2: {}, 3: {}, 4: {}}       # next_word[n][context] = {next word: count}
start_counts = Counter()
for idx, tokens in enumerate(stream_sentences(CORPUS_PATH)):
    if idx % EVERY != 0:
        continue
    tokens = [sys.intern(t) for t in tokens]
    start_counts[tokens[0]] += 1
    for n in (2, 3, 4):
        table = next_word[n]
        for i in range(len(tokens) - n + 1):
            ctx = tuple(tokens[i:i+n-1])
            d = table.get(ctx)
            if d is None:
                d = table[ctx] = {}
            w = tokens[i+n-1]
            d[w] = d.get(w, 0) + 1

start_words, start_weights = list(start_counts), list(start_counts.values())

def generate(n, max_len=20):
    words = random.choices(start_words, weights=start_weights)
    while len(words) < max_len:
        k = min(n, len(words) + 1)
        d = next_word[k].get(tuple(words[-(k-1):]))
        if not d:
            break
        words.append(random.choices(list(d), weights=list(d.values()))[0])
    return " ".join(words) + "።"

random.seed(42)
generated = []
for n, name in [(2, "Bigram"), (3, "Trigram"), (4, "Four-gram")]:
    print(f"\n=== {name} model ===")
    for i in range(5):
        s = generate(n)
        generated.append((name, i + 1, s))
        print(f"  {i+1}. {s}")

### Are generated sentences copied from the corpus? (checked against the full corpus)

In [ ]:
gen_words = [s.rstrip("።") for _, _, s in generated]
exact = [0] * len(gen_words)
part = [0] * len(gen_words)
for tokens in stream_sentences(CORPUS_PATH):
    line = " " + " ".join(tokens) + " "
    for j, g in enumerate(gen_words):
        if " " + g + " " in line:
            if line.strip() == g:
                exact[j] += 1
            else:
                part[j] += 1

rows = []
for (name, i, s), e, p in zip(generated, exact, part):
    status = "COPIED (exact sentence)" if e else "COPIED (part of a sentence)" if p else "NEW"
    rows.append([name, i, status, e, p, s])

for name in ["Bigram", "Trigram", "Four-gram"]:
    mine = [r for r in rows if r[0] == name]
    print(f"\n=== {name} model: {sum(r[2] != 'NEW' for r in mine)} of 5 copied ===")
    for r in mine:
        print(f"  {r[1]}. [{r[2]}] {r[5][:55]}...")

with open(OUT / "generated_sentences_full.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["model", "number", "status", "exact_matches", "partial_matches", "sentence"])
    w.writerows(rows)